In [17]:
import os
os.system('pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git" -q')
os.system('pip install "transformers==5.3.0" -q')
os.system('pip install "trl>=0.18.2,<=0.24.0,!=0.19.0" -q')
os.system('pip install rouge-score bert-score nltk -q')
os.system('pip install sentence-transformers -q')

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

True

In [18]:
import torch
import time
import json
import re
import numpy as np
import pandas as pd
from unsloth import FastLanguageModel
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from bert_score import score as bert_score

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


GPU: Tesla T4
VRAM: 15.6 GB


In [ ]:
# 12 functions:
#   1. search_products(category, color?)
#   2. check_product_availability(product_name, color, size)
#   3. create_order(product_name, color, size, quantity)
#   4. confirm_payment(payment_method)
#   5. add_delivery_details(name, phone, city, post_office)
#   6. confirm_order()
#   7. get_current_order_context()
#   8. track_order(order_id)
#   9. cancel_order(order_id)
#  10. update_order_item(order_id, field, new_value)
#  11. get_order_history(limit)
#  12. create_return_request(order_id, reason)
#
# Catalog:
#   dresses: Luna, Stella, Aurora, Nicol, Summer, Verona, Sofia
#   tops:    Vega, Cloud, Silk, Cotton, Breeze, Urban
#   jeans:   Classic, Slim, Straight, Relaxed
#   skirts:  Star, Midi, Pleat, Wrap
#   shoes:   Comfort, Sport, Elegant, Casual

import json as _json

def _tc(name, args):
    """Helper: assistant tool_call message у форматі тренування."""
    return {"role": "assistant",
            "content": f"<tool_call>{_json.dumps({'name': name, 'arguments': args})}</tool_call>"}

def _tr(content):
    """Helper: tool_response → user message (як у convert_ecommerce)."""
    return {"role": "user", "content": _json.dumps(content)}


TEST_CASES = [
    # 1. search_products  (first-turn) — 4 tests
    {"id": 1, "category": "search_products",
     "input": "What dresses do you have?",
     "expected_function": "search_products",
     "expected_params": {"category": "dresses"},
     "reference_response": "Here is what we have in dresses."},
    {"id": 2, "category": "search_products",
     "input": "Show me your jeans collection.",
     "expected_function": "search_products",
     "expected_params": {"category": "jeans"},
     "reference_response": "Here are our jeans."},
    {"id": 3, "category": "search_products",
     "input": "Show me tops in Black.",
     "expected_function": "search_products",
     "expected_params": {"category": "tops", "color": "Black"},
     "reference_response": "Here are black tops."},
    {"id": 4, "category": "search_products",
     "input": "What shoes are available?",
     "expected_function": "search_products",
     "expected_params": {"category": "shoes"},
     "reference_response": "Here are our shoes."},

    # 2. check_product_availability  (first-turn) — 4 tests
    {"id": 5, "category": "check_product_availability",
     "input": "I want to order the Luna dress in Black size M.",
     "expected_function": "check_product_availability",
     "expected_params": {"product_name": "Luna", "color": "Black", "size": "M"},
     "reference_response": "Let me check Luna in Black size M."},
    {"id": 6, "category": "check_product_availability",
     "input": "I'd like to buy the Vega top in White size S.",
     "expected_function": "check_product_availability",
     "expected_params": {"product_name": "Vega", "color": "White", "size": "S"},
     "reference_response": "Let me check Vega in White size S."},
    {"id": 7, "category": "check_product_availability",
     "input": "Can I get Classic jeans in Blue size 30?",
     "expected_function": "check_product_availability",
     "expected_params": {"product_name": "Classic", "color": "Blue", "size": "30"},
     "reference_response": "Let me check Classic in Blue size 30."},
    {"id": 8, "category": "check_product_availability",
     "input": "I need Star skirt in Red size S.",
     "expected_function": "check_product_availability",
     "expected_params": {"product_name": "Star", "color": "Red", "size": "S"},
     "reference_response": "Let me check Star in Red size S."},

    # 3. create_order  (multi-turn: after availability + user agrees) — 4 tests
    {"id": 9, "category": "create_order",
     "turns": [
         {"role": "user", "content": "I want to buy Luna dress in Black size M"},
         _tc("check_product_availability", {"product_name": "Luna", "color": "Black", "size": "M"}),
         _tr({"available": True, "stock_quantity": 5, "price": 49.99}),
         {"role": "assistant", "content": "Luna in Black size M is available for $49.99! Ready to proceed?"},
         {"role": "user", "content": "Yes please"},
     ],
     "expected_function": "create_order",
     "expected_params": {"product_name": "Luna", "color": "Black", "size": "M", "quantity": 1},
     "reference_response": "Creating your order for Luna."},
    {"id": 10, "category": "create_order",
     "turns": [
         {"role": "user", "content": "I'd like Aurora dress in Pink size S"},
         _tc("check_product_availability", {"product_name": "Aurora", "color": "Pink", "size": "S"}),
         _tr({"available": True, "stock_quantity": 8, "price": 54.99}),
         {"role": "assistant", "content": "Aurora in Pink size S is available! Want to order?"},
         {"role": "user", "content": "Sure, let's go"},
     ],
     "expected_function": "create_order",
     "expected_params": {"product_name": "Aurora", "color": "Pink", "size": "S", "quantity": 1},
     "reference_response": "Creating your order."},
    {"id": 11, "category": "create_order",
     "turns": [
         {"role": "user", "content": "I want Sport shoes in White size 38"},
         _tc("check_product_availability", {"product_name": "Sport", "color": "White", "size": "38"}),
         _tr({"available": True, "stock_quantity": 3, "price": 79.99}),
         {"role": "assistant", "content": "Sport shoes White size 38 are in stock for $79.99. Proceed?"},
         {"role": "user", "content": "Confirmed"},
     ],
     "expected_function": "create_order",
     "expected_params": {"product_name": "Sport", "color": "White", "size": "38", "quantity": 1},
     "reference_response": "Order being created."},
    {"id": 12, "category": "create_order",
     "turns": [
         {"role": "user", "content": "I'd like Classic jeans in Blue size 30"},
         _tc("check_product_availability", {"product_name": "Classic", "color": "Blue", "size": "30"}),
         _tr({"available": True, "stock_quantity": 6, "price": 69.99}),
         {"role": "assistant", "content": "Classic jeans in Blue size 30 are available for $69.99! Shall I order them?"},
         {"role": "user", "content": "Yes, order it"},
     ],
     "expected_function": "create_order",
     "expected_params": {"product_name": "Classic", "color": "Blue", "size": "30", "quantity": 1},
     "reference_response": "Creating your order for Classic jeans."},

    # 4. confirm_payment  (multi-turn: after order + payment method) — 4 tests
    {"id": 13, "category": "confirm_payment",
     "turns": [
         {"role": "user", "content": "I want Luna dress Black size M"},
         _tc("check_product_availability", {"product_name": "Luna", "color": "Black", "size": "M"}),
         _tr({"available": True, "stock_quantity": 5, "price": 49.99}),
         {"role": "assistant", "content": "Available! Ready to proceed?"},
         {"role": "user", "content": "Yes"},
         _tc("create_order", {"product_name": "Luna", "color": "Black", "size": "M", "quantity": 1}),
         _tr({"order_id": 1234, "status": "draft"}),
         {"role": "assistant", "content": "Order created! How would you like to pay? (card/cash)"},
         {"role": "user", "content": "I'll pay by card"},
     ],
     "expected_function": "confirm_payment",
     "expected_params": {"payment_method": "card"},
     "reference_response": "Confirming card payment."},
    {"id": 14, "category": "confirm_payment",
     "turns": [
         {"role": "user", "content": "Aurora dress Pink size S"},
         _tc("check_product_availability", {"product_name": "Aurora", "color": "Pink", "size": "S"}),
         _tr({"available": True, "stock_quantity": 4, "price": 54.99}),
         {"role": "assistant", "content": "Available! Proceed?"},
         {"role": "user", "content": "Yes please"},
         _tc("create_order", {"product_name": "Aurora", "color": "Pink", "size": "S", "quantity": 1}),
         _tr({"order_id": 5678, "status": "draft"}),
         {"role": "assistant", "content": "Order ready. Card or cash?"},
         {"role": "user", "content": "Cash on delivery"},
     ],
     "expected_function": "confirm_payment",
     "expected_params": {"payment_method": "cash"},
     "reference_response": "Confirming cash payment."},
    {"id": 15, "category": "confirm_payment",
     "turns": [
         {"role": "user", "content": "I want Vega top White size S"},
         _tc("check_product_availability", {"product_name": "Vega", "color": "White", "size": "S"}),
         _tr({"available": True, "stock_quantity": 9, "price": 29.99}),
         {"role": "assistant", "content": "Vega in White size S is available! Proceed?"},
         {"role": "user", "content": "Yes"},
         _tc("create_order", {"product_name": "Vega", "color": "White", "size": "S", "quantity": 1}),
         _tr({"order_id": 4400, "status": "draft"}),
         {"role": "assistant", "content": "Order created! How would you like to pay?"},
         {"role": "user", "content": "By card please"},
     ],
     "expected_function": "confirm_payment",
     "expected_params": {"payment_method": "card"},
     "reference_response": "Confirming card payment."},
    {"id": 16, "category": "confirm_payment",
     "turns": [
         {"role": "user", "content": "Comfort shoes Black size 39"},
         _tc("check_product_availability", {"product_name": "Comfort", "color": "Black", "size": "39"}),
         _tr({"available": True, "stock_quantity": 2, "price": 89.99}),
         {"role": "assistant", "content": "Comfort shoes Black size 39 available! Proceed?"},
         {"role": "user", "content": "Yes, order"},
         _tc("create_order", {"product_name": "Comfort", "color": "Black", "size": "39", "quantity": 1}),
         _tr({"order_id": 7100, "status": "draft"}),
         {"role": "assistant", "content": "Order ready. Card or cash?"},
         {"role": "user", "content": "I'll pay cash"},
     ],
     "expected_function": "confirm_payment",
     "expected_params": {"payment_method": "cash"},
     "reference_response": "Confirming cash payment."},

    # 5. add_delivery_details  (multi-turn: after payment) — 4 tests
    {"id": 17, "category": "add_delivery_details",
     "turns": [
         {"role": "user", "content": "Luna dress Black size M"},
         _tc("check_product_availability", {"product_name": "Luna", "color": "Black", "size": "M"}),
         _tr({"available": True, "stock_quantity": 5, "price": 49.99}),
         {"role": "assistant", "content": "Available, proceed?"},
         {"role": "user", "content": "Yes"},
         _tc("create_order", {"product_name": "Luna", "color": "Black", "size": "M", "quantity": 1}),
         _tr({"order_id": 1234, "status": "draft"}),
         {"role": "assistant", "content": "How would you like to pay?"},
         {"role": "user", "content": "Card"},
         _tc("confirm_payment", {"payment_method": "card"}),
         _tr({"status": "paid", "transaction_id": "TXN_001"}),
         {"role": "assistant", "content": "Payment confirmed! Please provide name, phone, city and post office."},
         {"role": "user", "content": "Name: Anna, Phone: +380501234567, City: Kyiv, Post office: 5"},
     ],
     "expected_function": "add_delivery_details",
     "expected_params": {"name": "Anna", "city": "Kyiv", "post_office": "5"},
     "reference_response": "Adding delivery details."},
    {"id": 18, "category": "add_delivery_details",
     "turns": [
         {"role": "user", "content": "I want Vega top White size S"},
         _tc("check_product_availability", {"product_name": "Vega", "color": "White", "size": "S"}),
         _tr({"available": True, "stock_quantity": 7, "price": 29.99}),
         {"role": "assistant", "content": "Available!"},
         {"role": "user", "content": "Order it"},
         _tc("create_order", {"product_name": "Vega", "color": "White", "size": "S", "quantity": 1}),
         _tr({"order_id": 9876, "status": "draft"}),
         {"role": "assistant", "content": "Card or cash?"},
         {"role": "user", "content": "Card"},
         _tc("confirm_payment", {"payment_method": "card"}),
         _tr({"status": "paid", "transaction_id": "TXN_002"}),
         {"role": "assistant", "content": "Please share delivery info."},
         {"role": "user", "content": "Maria, +380671112233, Lviv, post 12"},
     ],
     "expected_function": "add_delivery_details",
     "expected_params": {"name": "Maria", "city": "Lviv", "post_office": "12"},
     "reference_response": "Saving delivery details."},
    {"id": 19, "category": "add_delivery_details",
     "turns": [
         {"role": "user", "content": "Classic jeans Blue size 32"},
         _tc("check_product_availability", {"product_name": "Classic", "color": "Blue", "size": "32"}),
         _tr({"available": True, "stock_quantity": 4, "price": 69.99}),
         {"role": "assistant", "content": "Available, proceed?"},
         {"role": "user", "content": "Yes"},
         _tc("create_order", {"product_name": "Classic", "color": "Blue", "size": "32", "quantity": 1}),
         _tr({"order_id": 3300, "status": "draft"}),
         {"role": "assistant", "content": "How would you like to pay?"},
         {"role": "user", "content": "Cash"},
         _tc("confirm_payment", {"payment_method": "cash"}),
         _tr({"status": "paid", "transaction_id": "TXN_003"}),
         {"role": "assistant", "content": "Please provide your delivery details."},
         {"role": "user", "content": "Olena, +380631234567, Odesa, post office 8"},
     ],
     "expected_function": "add_delivery_details",
     "expected_params": {"name": "Olena", "city": "Odesa", "post_office": "8"},
     "reference_response": "Adding delivery details."},
    {"id": 20, "category": "add_delivery_details",
     "turns": [
         {"role": "user", "content": "Sport shoes Blue size 40"},
         _tc("check_product_availability", {"product_name": "Sport", "color": "Blue", "size": "40"}),
         _tr({"available": True, "stock_quantity": 3, "price": 79.99}),
         {"role": "assistant", "content": "Available!"},
         {"role": "user", "content": "Order it"},
         _tc("create_order", {"product_name": "Sport", "color": "Blue", "size": "40", "quantity": 1}),
         _tr({"order_id": 8200, "status": "draft"}),
         {"role": "assistant", "content": "Card or cash?"},
         {"role": "user", "content": "Card"},
         _tc("confirm_payment", {"payment_method": "card"}),
         _tr({"status": "paid", "transaction_id": "TXN_004"}),
         {"role": "assistant", "content": "Please share your delivery info."},
         {"role": "user", "content": "John, +380991234567, Kharkiv, post 3"},
     ],
     "expected_function": "add_delivery_details",
     "expected_params": {"name": "John", "city": "Kharkiv", "post_office": "3"},
     "reference_response": "Saving delivery details."},

    # 6. confirm_order  (multi-turn: AFTER add_delivery_details) — 4 tests
    {"id": 21, "category": "confirm_order",
     "turns": [
         {"role": "user", "content": "Luna dress Black size M"},
         _tc("check_product_availability", {"product_name": "Luna", "color": "Black", "size": "M"}),
         _tr({"available": True, "stock_quantity": 5, "price": 49.99}),
         {"role": "assistant", "content": "Available, proceed?"},
         {"role": "user", "content": "Yes"},
         _tc("create_order", {"product_name": "Luna", "color": "Black", "size": "M", "quantity": 1}),
         _tr({"order_id": 1234, "status": "draft"}),
         {"role": "assistant", "content": "Card or cash?"},
         {"role": "user", "content": "Card"},
         _tc("confirm_payment", {"payment_method": "card"}),
         _tr({"status": "paid", "transaction_id": "TXN_001"}),
         {"role": "assistant", "content": "Please share delivery info."},
         {"role": "user", "content": "Anna, +380501234567, Kyiv, post 5"},
         _tc("add_delivery_details", {"name": "Anna", "phone": "+380501234567",
                                       "city": "Kyiv", "post_office": "5"}),
         _tr({"delivery_id": "DEL_500"}),
     ],
     "expected_function": "confirm_order",
     "expected_params": {},
     "reference_response": "Confirming your order."},
    {"id": 22, "category": "confirm_order",
     "turns": [
         {"role": "user", "content": "Aurora dress Pink size S"},
         _tc("check_product_availability", {"product_name": "Aurora", "color": "Pink", "size": "S"}),
         _tr({"available": True, "stock_quantity": 8, "price": 54.99}),
         {"role": "assistant", "content": "Available, proceed?"},
         {"role": "user", "content": "Yes"},
         _tc("create_order", {"product_name": "Aurora", "color": "Pink", "size": "S", "quantity": 1}),
         _tr({"order_id": 5678, "status": "draft"}),
         {"role": "assistant", "content": "Card or cash?"},
         {"role": "user", "content": "Cash"},
         _tc("confirm_payment", {"payment_method": "cash"}),
         _tr({"status": "paid", "transaction_id": "TXN_002"}),
         {"role": "assistant", "content": "Please share delivery info."},
         {"role": "user", "content": "Maria, +380671112233, Lviv, post 12"},
         _tc("add_delivery_details", {"name": "Maria", "phone": "+380671112233",
                                       "city": "Lviv", "post_office": "12"}),
         _tr({"delivery_id": "DEL_501"}),
     ],
     "expected_function": "confirm_order",
     "expected_params": {},
     "reference_response": "Confirming your order."},
    {"id": 23, "category": "confirm_order",
     "turns": [
         {"role": "user", "content": "Classic jeans Blue size 30"},
         _tc("check_product_availability", {"product_name": "Classic", "color": "Blue", "size": "30"}),
         _tr({"available": True, "stock_quantity": 6, "price": 69.99}),
         {"role": "assistant", "content": "Available, proceed?"},
         {"role": "user", "content": "Yes"},
         _tc("create_order", {"product_name": "Classic", "color": "Blue", "size": "30", "quantity": 1}),
         _tr({"order_id": 3301, "status": "draft"}),
         {"role": "assistant", "content": "Card or cash?"},
         {"role": "user", "content": "Card"},
         _tc("confirm_payment", {"payment_method": "card"}),
         _tr({"status": "paid", "transaction_id": "TXN_003"}),
         {"role": "assistant", "content": "Please share delivery info."},
         {"role": "user", "content": "Olena, +380631234567, Odesa, post 8"},
         _tc("add_delivery_details", {"name": "Olena", "phone": "+380631234567",
                                       "city": "Odesa", "post_office": "8"}),
         _tr({"delivery_id": "DEL_502"}),
     ],
     "expected_function": "confirm_order",
     "expected_params": {},
     "reference_response": "Confirming your order."},
    {"id": 24, "category": "confirm_order",
     "turns": [
         {"role": "user", "content": "Vega top Blue size M"},
         _tc("check_product_availability", {"product_name": "Vega", "color": "Blue", "size": "M"}),
         _tr({"available": True, "stock_quantity": 5, "price": 29.99}),
         {"role": "assistant", "content": "Available, proceed?"},
         {"role": "user", "content": "Yes"},
         _tc("create_order", {"product_name": "Vega", "color": "Blue", "size": "M", "quantity": 1}),
         _tr({"order_id": 4401, "status": "draft"}),
         {"role": "assistant", "content": "Card or cash?"},
         {"role": "user", "content": "Card"},
         _tc("confirm_payment", {"payment_method": "card"}),
         _tr({"status": "paid", "transaction_id": "TXN_004"}),
         {"role": "assistant", "content": "Please share delivery info."},
         {"role": "user", "content": "John, +380991234567, Kharkiv, post 3"},
         _tc("add_delivery_details", {"name": "John", "phone": "+380991234567",
                                       "city": "Kharkiv", "post_office": "3"}),
         _tr({"delivery_id": "DEL_503"}),
     ],
     "expected_function": "confirm_order",
     "expected_params": {},
     "reference_response": "Confirming your order."},

    # 7. get_current_order_context  (first-turn) — 4 tests
    {"id": 25, "category": "get_current_order_context",
     "input": "Where is my order? When will it arrive?",
     "expected_function": "get_current_order_context",
     "expected_params": {},
     "reference_response": "Let me check your order status."},
    {"id": 26, "category": "get_current_order_context",
     "input": "Please cancel my order, I changed my mind.",
     "expected_function": "get_current_order_context",
     "expected_params": {},
     "reference_response": "Let me find your order to cancel."},
    {"id": 27, "category": "get_current_order_context",
     "input": "Change size to L please.",
     "expected_function": "get_current_order_context",
     "expected_params": {},
     "reference_response": "Let me find your order to update size."},
    {"id": 28, "category": "get_current_order_context",
     "input": "Has my order been shipped yet?",
     "expected_function": "get_current_order_context",
     "expected_params": {},
     "reference_response": "Let me check your shipping status."},

    # 8. track_order  (multi-turn: after get_current_order_context) — 4 tests
    {"id": 29, "category": "track_order",
     "turns": [
         {"role": "user", "content": "Where is my order?"},
         _tc("get_current_order_context", {}),
         _tr({"active_orders": [{"order_id": 5678, "product": "Luna dress",
                                  "size": "M", "color": "Black", "status": "shipped"}]}),
     ],
     "expected_function": "track_order",
     "expected_params": {"order_id": 5678},
     "reference_response": "Tracking your order."},
    {"id": 30, "category": "track_order",
     "turns": [
         {"role": "user", "content": "Track my delivery please."},
         _tc("get_current_order_context", {}),
         _tr({"active_orders": [{"order_id": 9123, "product": "Vega top",
                                  "size": "S", "color": "White", "status": "out_for_delivery"}]}),
     ],
     "expected_function": "track_order",
     "expected_params": {"order_id": 9123},
     "reference_response": "Tracking now."},
    {"id": 31, "category": "track_order",
     "turns": [
         {"role": "user", "content": "What's the status of my order?"},
         _tc("get_current_order_context", {}),
         _tr({"active_orders": [{"order_id": 6200, "product": "Classic jeans",
                                  "size": "30", "color": "Blue", "status": "processing"}]}),
     ],
     "expected_function": "track_order",
     "expected_params": {"order_id": 6200},
     "reference_response": "Checking your order status."},
    {"id": 32, "category": "track_order",
     "turns": [
         {"role": "user", "content": "Has my package been sent?"},
         _tc("get_current_order_context", {}),
         _tr({"active_orders": [{"order_id": 7350, "product": "Sport shoes",
                                  "size": "39", "color": "Black", "status": "shipped"}]}),
     ],
     "expected_function": "track_order",
     "expected_params": {"order_id": 7350},
     "reference_response": "Tracking your package."},

    # 9. cancel_order  (multi-turn: after get_current_order_context) — 4 tests
    {"id": 33, "category": "cancel_order",
     "turns": [
         {"role": "user", "content": "Cancel my order please"},
         _tc("get_current_order_context", {}),
         _tr({"active_orders": [{"order_id": 7777, "product": "Aurora dress",
                                  "size": "S", "color": "Pink", "status": "draft"}]}),
         {"role": "assistant", "content": "Found your order for Aurora dress (Pink, size S). Cancelling now..."},
     ],
     "expected_function": "cancel_order",
     "expected_params": {"order_id": 7777},
     "reference_response": "Cancelling order."},
    {"id": 34, "category": "cancel_order",
     "turns": [
         {"role": "user", "content": "I don't want it anymore, cancel it"},
         _tc("get_current_order_context", {}),
         _tr({"active_orders": [{"order_id": 4321, "product": "Sport shoes",
                                  "size": "38", "color": "White", "status": "paid"}]}),
         {"role": "assistant", "content": "Found your Sport shoes order. Cancelling..."},
     ],
     "expected_function": "cancel_order",
     "expected_params": {"order_id": 4321},
     "reference_response": "Cancelling now."},
    {"id": 35, "category": "cancel_order",
     "turns": [
         {"role": "user", "content": "Please cancel that order"},
         _tc("get_current_order_context", {}),
         _tr({"active_orders": [{"order_id": 5150, "product": "Luna dress",
                                  "size": "L", "color": "Red", "status": "processing"}]}),
         {"role": "assistant", "content": "Found your Luna dress order. Cancelling it now..."},
     ],
     "expected_function": "cancel_order",
     "expected_params": {"order_id": 5150},
     "reference_response": "Cancelling order."},
    {"id": 36, "category": "cancel_order",
     "turns": [
         {"role": "user", "content": "Cancel it, I made a mistake"},
         _tc("get_current_order_context", {}),
         _tr({"active_orders": [{"order_id": 8800, "product": "Classic jeans",
                                  "size": "32", "color": "Black", "status": "draft"}]}),
         {"role": "assistant", "content": "Found your Classic jeans order. Cancelling..."},
     ],
     "expected_function": "cancel_order",
     "expected_params": {"order_id": 8800},
     "reference_response": "Cancelling now."},

    # 10. update_order_item  (multi-turn: after get_current_order_context) — 4 tests
    {"id": 37, "category": "update_order_item",
     "turns": [
         {"role": "user", "content": "Change size to L"},
         _tc("get_current_order_context", {}),
         _tr({"active_orders": [{"order_id": 8800, "product": "Luna dress",
                                  "size": "M", "color": "Black", "status": "confirmed"}]}),
         {"role": "assistant", "content": "Found your order. Updating size to L..."},
     ],
     "expected_function": "update_order_item",
     "expected_params": {"order_id": 8800, "field": "size", "new_value": "L"},
     "reference_response": "Updating size."},
    {"id": 38, "category": "update_order_item",
     "turns": [
         {"role": "user", "content": "Change the color to White please"},
         _tc("get_current_order_context", {}),
         _tr({"active_orders": [{"order_id": 3333, "product": "Vega top",
                                  "size": "M", "color": "Black", "status": "confirmed"}]}),
         {"role": "assistant", "content": "Found your Vega top order. Updating color to White..."},
     ],
     "expected_function": "update_order_item",
     "expected_params": {"order_id": 3333, "field": "color", "new_value": "White"},
     "reference_response": "Updating color."},
    {"id": 39, "category": "update_order_item",
     "turns": [
         {"role": "user", "content": "Can you change the size to XL?"},
         _tc("get_current_order_context", {}),
         _tr({"active_orders": [{"order_id": 6700, "product": "Nicol dress",
                                  "size": "L", "color": "Beige", "status": "confirmed"}]}),
         {"role": "assistant", "content": "Found your Nicol dress order. Updating size to XL..."},
     ],
     "expected_function": "update_order_item",
     "expected_params": {"order_id": 6700, "field": "size", "new_value": "XL"},
     "reference_response": "Updating size."},
    {"id": 40, "category": "update_order_item",
     "turns": [
         {"role": "user", "content": "I'd like to change the color to Black"},
         _tc("get_current_order_context", {}),
         _tr({"active_orders": [{"order_id": 2100, "product": "Classic jeans",
                                  "size": "30", "color": "Blue", "status": "confirmed"}]}),
         {"role": "assistant", "content": "Found your Classic jeans order. Updating color to Black..."},
     ],
     "expected_function": "update_order_item",
     "expected_params": {"order_id": 2100, "field": "color", "new_value": "Black"},
     "reference_response": "Updating color."},

    # 11. get_order_history  (first-turn) — 4 tests
    {"id": 41, "category": "get_order_history",
     "input": "Show me my orders.",
     "expected_function": "get_order_history",
     "expected_params": {"limit": 10},
     "reference_response": "Here is your order history."},
    {"id": 42, "category": "get_order_history",
     "input": "What are my previous orders?",
     "expected_function": "get_order_history",
     "expected_params": {"limit": 10},
     "reference_response": "Here are your previous orders."},
    {"id": 43, "category": "get_order_history",
     "input": "What have I ordered before?",
     "expected_function": "get_order_history",
     "expected_params": {"limit": 10},
     "reference_response": "Here is your purchase history."},
    {"id": 44, "category": "get_order_history",
     "input": "Can I see my order history?",
     "expected_function": "get_order_history",
     "expected_params": {"limit": 10},
     "reference_response": "Here is your order history."},

    # 12. create_return_request  (multi-turn: after get_order_history) — 4 tests
    {"id": 45, "category": "create_return_request",
     "turns": [
         {"role": "user", "content": "I want to return my order"},
         _tc("get_order_history", {"limit": 5}),
         _tr({"orders": [{"order_id": 1111, "product": "Luna dress",
                          "color": "Black", "size": "M",
                          "status": "delivered", "date": "2026-04-20"}]}),
         {"role": "assistant", "content": "Found your Luna dress order. What is the reason for the return?"},
         {"role": "user", "content": "doesn't fit"},
     ],
     "expected_function": "create_return_request",
     "expected_params": {"order_id": 1111, "reason": "doesn't fit"},
     "reference_response": "Initiating return."},
    {"id": 46, "category": "create_return_request",
     "turns": [
         {"role": "user", "content": "I want a refund"},
         _tc("get_order_history", {"limit": 5}),
         _tr({"orders": [{"order_id": 2222, "product": "Vega top",
                          "color": "White", "size": "S",
                          "status": "delivered", "date": "2026-04-18"}]}),
         {"role": "assistant", "content": "Found Vega top. What's the reason?"},
         {"role": "user", "content": "item is damaged"},
     ],
     "expected_function": "create_return_request",
     "expected_params": {"order_id": 2222, "reason": "item is damaged"},
     "reference_response": "Creating return request."},
    {"id": 47, "category": "create_return_request",
     "turns": [
         {"role": "user", "content": "I'd like to return something I bought"},
         _tc("get_order_history", {"limit": 5}),
         _tr({"orders": [{"order_id": 3333, "product": "Classic jeans",
                          "color": "Blue", "size": "30",
                          "status": "delivered", "date": "2026-04-15"}]}),
         {"role": "assistant", "content": "Found your Classic jeans order. What is the reason for the return?"},
         {"role": "user", "content": "wrong size"},
     ],
     "expected_function": "create_return_request",
     "expected_params": {"order_id": 3333, "reason": "wrong size"},
     "reference_response": "Initiating return."},
    {"id": 48, "category": "create_return_request",
     "turns": [
         {"role": "user", "content": "Can I send back my order?"},
         _tc("get_order_history", {"limit": 5}),
         _tr({"orders": [{"order_id": 4444, "product": "Sport shoes",
                          "color": "Black", "size": "39",
                          "status": "delivered", "date": "2026-04-12"}]}),
         {"role": "assistant", "content": "Found your Sport shoes order. What's the reason for the return?"},
         {"role": "user", "content": "changed my mind"},
     ],
     "expected_function": "create_return_request",
     "expected_params": {"order_id": 4444, "reason": "changed my mind"},
     "reference_response": "Creating return request."},

    # CUSTOMER SUPPORT  (NO tool call expected — only support) — 14 tests
    {"id": 49, "category": "customer_support",
     "input": "What is your return policy?",
     "expected_function": None, "expected_params": {},
     "reference_response": "Our return policy allows returns within 30 days."},
    {"id": 50, "category": "customer_support",
     "input": "How long does shipping take?",
     "expected_function": None, "expected_params": {},
     "reference_response": "Standard shipping takes 3-7 business days."},
    {"id": 51, "category": "customer_support",
     "input": "Do you offer free shipping?",
     "expected_function": None, "expected_params": {},
     "reference_response": "We offer free shipping on orders over $50."},
    {"id": 52, "category": "customer_support",
     "input": "My payment was declined. What should I do?",
     "expected_function": None, "expected_params": {},
     "reference_response": "Please check your card details."},
    {"id": 53, "category": "customer_support",
     "input": "I haven't received my refund yet.",
     "expected_function": None, "expected_params": {},
     "reference_response": "Refunds typically take 5-10 business days."},
    {"id": 54, "category": "customer_support",
     "input": "How can I contact customer support?",
     "expected_function": None, "expected_params": {},
     "reference_response": "You can email us or call us."},
    {"id": 55, "category": "customer_support",
     "input": "What payment methods do you accept?",
     "expected_function": None, "expected_params": {},
     "reference_response": "We accept card and cash on delivery."},
    {"id": 56, "category": "customer_support",
     "input": "Do you have a size guide?",
     "expected_function": None, "expected_params": {},
     "reference_response": "Yes, our size guide is available on each product page."},
    {"id": 57, "category": "customer_support",
     "input": "Can I change my account email?",
     "expected_function": None, "expected_params": {},
     "reference_response": "You can update your email in account settings."},
    {"id": 58, "category": "customer_support",
     "input": "What are your working hours?",
     "expected_function": None, "expected_params": {},
     "reference_response": "Our support team works 9am to 6pm on weekdays."},
    {"id": 59, "category": "customer_support",
     "input": "Do you ship internationally?",
     "expected_function": None, "expected_params": {},
     "reference_response": "Currently we ship within the country only."},
    {"id": 60, "category": "customer_support",
     "input": "Is there a warranty on your products?",
     "expected_function": None, "expected_params": {},
     "reference_response": "All items come with a standard quality guarantee."},
    {"id": 61, "category": "customer_support",
     "input": "How do I create an account?",
     "expected_function": None, "expected_params": {},
     "reference_response": "You can sign up using your email on our website."},
    {"id": 62, "category": "customer_support",
     "input": "Do you offer gift cards?",
     "expected_function": None, "expected_params": {},
     "reference_response": "Yes, gift cards are available in various amounts."},
]

CONSISTENCY_CASES = [
    {"input": "Where is my order?",                 "expected_function": "get_current_order_context"},
    {"input": "Cancel my order please",             "expected_function": "get_current_order_context"},
    {"input": "What dresses do you have?",          "expected_function": "search_products"},
    {"input": "I want to return my order",          "expected_function": "get_order_history"},
    {"input": "Show me my orders",                  "expected_function": "get_order_history"},
    {"input": "I want Luna dress in Black size M",  "expected_function": "check_product_availability"},
    {"input": "Show me your shoes",                 "expected_function": "search_products"},
    {"input": "Track my delivery",                  "expected_function": "get_current_order_context"},
    {"input": "I'd like to buy Vega top in White size S", "expected_function": "check_product_availability"},
    {"input": "What have I ordered before?",        "expected_function": "get_order_history"},
]

OUT_OF_SCOPE_CASES = [
    {"input": "What is the capital of France?"},
    {"input": "Write me a poem about flowers"},
    {"input": "What is 2 + 2?"},
    {"input": "Who is Elon Musk?"},
    {"input": "Tell me a joke"},
    {"input": "What is the weather today?"},
    {"input": "Translate 'hello' to Spanish"},
    {"input": "How do I cook pasta?"},
    {"input": "What year did World War II end?"},
    {"input": "Recommend me a good movie"},
    {"input": "Explain how photosynthesis works"},
    {"input": "What is the meaning of life?"},
]

MULTITURN_CASES = [
    {"id": "MT1",
     "turns": [
         {"role": "user", "content": "Hi, I want to buy a Luna dress"},
         {"role": "assistant", "content": "Sure! What color and size?"},
         {"role": "user", "content": "Black, size M please"},
     ],
     "expected_function": "check_product_availability"},
    {"id": "MT2",
     "turns": [
         {"role": "user", "content": "Show me dresses"},
         {"role": "assistant", "content": "Here are our dresses: Luna, Stella, Aurora..."},
         {"role": "user", "content": "I like Luna, do you have it in Black size M?"},
     ],
     "expected_function": "check_product_availability"},
    {"id": "MT3",
     "turns": [
         {"role": "user", "content": "I want to return something"},
         {"role": "assistant", "content": "Sure, let me check your orders."},
         {"role": "user", "content": "Yes please"},
     ],
     "expected_function": "get_order_history"},
    {"id": "MT4",
     "turns": [
         {"role": "user", "content": "I'm looking for a top"},
         {"role": "assistant", "content": "Great! What color would you like?"},
         {"role": "user", "content": "White, size S — the Vega one"},
     ],
     "expected_function": "check_product_availability"},
    {"id": "MT5",
     "turns": [
         {"role": "user", "content": "Do you sell jeans?"},
         {"role": "assistant", "content": "Yes! Here are our jeans: Classic, Slim, Straight, Relaxed."},
         {"role": "user", "content": "Classic in Blue size 30, is it in stock?"},
     ],
     "expected_function": "check_product_availability"},
    {"id": "MT6",
     "turns": [
         {"role": "user", "content": "I need to check on my order"},
         {"role": "assistant", "content": "Sure, let me look that up for you."},
         {"role": "user", "content": "Yes, where is it?"},
     ],
     "expected_function": "get_current_order_context"},
    {"id": "MT7",
     "turns": [
         {"role": "user", "content": "Can you show me what's available?"},
         {"role": "assistant", "content": "Of course! Which category - dresses, tops, jeans, skirts or shoes?"},
         {"role": "user", "content": "Shoes please"},
     ],
     "expected_function": "search_products"},
    {"id": "MT8",
     "turns": [
         {"role": "user", "content": "I want to see my past purchases"},
         {"role": "assistant", "content": "Let me pull up your order history."},
         {"role": "user", "content": "Go ahead"},
     ],
     "expected_function": "get_order_history"},
]

EDGE_CASES = [
    {"id": "E1", "category": "edge_missing_info",
     "input": "I want to buy a dress.",
     "should_ask_clarification": True, "should_call_tool": False},
    {"id": "E2", "category": "edge_missing_info",
     "input": "I want to order something.",
     "should_ask_clarification": True, "should_call_tool": False},
    {"id": "E3", "category": "edge_ambiguous",
     "input": "I need help with my order.",
     "should_ask_clarification": True, "should_call_tool": False},
    {"id": "E4", "category": "edge_missing_info",
     "input": "I'd like to buy the Luna dress.",
     "should_ask_clarification": True, "should_call_tool": False},
    {"id": "E5", "category": "edge_missing_info",
     "input": "Can I get some shoes?",
     "should_ask_clarification": True, "should_call_tool": False},
    {"id": "E6", "category": "edge_missing_info",
     "input": "I want the Vega top in White.",
     "should_ask_clarification": True, "should_call_tool": False},
    {"id": "E7", "category": "edge_ambiguous",
     "input": "Can you help me?",
     "should_ask_clarification": True, "should_call_tool": False},
    {"id": "E8", "category": "edge_ambiguous",
     "input": "I have a problem.",
     "should_ask_clarification": True, "should_call_tool": False},
    {"id": "E9", "category": "edge_missing_info",
     "input": "I want jeans in size 30.",
     "should_ask_clarification": True, "should_call_tool": False},
    {"id": "E10", "category": "edge_ambiguous",
     "input": "Do something with my order.",
     "should_ask_clarification": True, "should_call_tool": False},
]

cats = {}
for tc in TEST_CASES:
    cats.setdefault(tc["category"], 0)
    cats[tc["category"]] += 1

print(f"{'='*60}")
print(f"TOTAL TEST CASES: {len(TEST_CASES)}")
print(f"{'='*60}")
for cat, cnt in cats.items():
    fmt = "first-turn" if any(("input" in t and t["category"] == cat) for t in TEST_CASES) else "multi-turn"
    print(f"  {cat:<32}: {cnt}  ({fmt})")
print(f"{'-'*60}")
print(f"  consistency                     : {len(CONSISTENCY_CASES)} x 3 runs")
print(f"  out_of_scope                    : {len(OUT_OF_SCOPE_CASES)}")
print(f"  multi_turn (extra)              : {len(MULTITURN_CASES)}")
print(f"  edge_cases                      : {len(EDGE_CASES)}")
print()


In [20]:
import math
import json
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from bert_score import score as bert_score
from sentence_transformers import SentenceTransformer, util

sem_model = SentenceTransformer('all-MiniLM-L6-v2')
print("Sentence transformer loaded!")

def parse_tool_call(response):
    if "<tool_call>" not in response:
        return None, None
    try:
        start = response.index("<tool_call>") + len("<tool_call>")
        end   = response.index("</tool_call>")
        data  = json.loads(response[start:end].strip())
        return data.get("name"), data.get("arguments", {})
    except:
        return None, None

def compute_rouge_l(hypothesis, reference):
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    return round(scorer.score(reference, hypothesis)['rougeL'].fmeasure, 4)

def compute_bleu4(hypothesis, reference):
    smoothie = SmoothingFunction().method1
    try:
        return round(sentence_bleu([reference.lower().split()], hypothesis.lower().split(),
                                    weights=(0.25,0.25,0.25,0.25), smoothing_function=smoothie), 4)
    except:
        return 0.0

def compute_perplexity(val_loss):
    return round(math.exp(val_loss), 4)

def compute_semantic_similarity(hypothesis, reference):
    emb1 = sem_model.encode(hypothesis, convert_to_tensor=True)
    emb2 = sem_model.encode(reference,  convert_to_tensor=True)
    return round(util.cos_sim(emb1, emb2).item(), 4)

def check_params(extracted, expected):
    if not expected: return 1.0
    if not extracted: return 0.0
    return round(sum(1 for k in expected if k in extracted) / len(expected), 4)

def evaluate_consistency(model, tokenizer, cases, n_runs=3):
    scores = []
    for tc in cases:
        fn_results = []
        for _ in range(n_runs):
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": tc["input"]}
            ]
            inputs = tokenizer.apply_chat_template(
                messages, tokenize=True,
                add_generation_prompt=True,
                return_tensors="pt"
            ).to("cuda")
            with torch.no_grad():
                outputs = model.generate(
                    input_ids=inputs, max_new_tokens=150,
                    temperature=0.1, do_sample=False,
                    pad_token_id=tokenizer.eos_token_id,
                )
            response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
            fn_name, _ = parse_tool_call(response)
            fn_results.append(fn_name == tc["expected_function"])
        consistency = sum(fn_results) / n_runs
        scores.append(consistency)
        print(f"  '{tc['input'][:40]}' → {fn_results} = {consistency:.0%}")
    return round(sum(scores) / len(scores), 4)

def evaluate_out_of_scope(model, tokenizer, cases):
    correct = 0
    for tc in cases:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": tc["input"]}
        ]
        inputs = tokenizer.apply_chat_template(
            messages, tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt"
        ).to("cuda")
        with torch.no_grad():
            outputs = model.generate(
                input_ids=inputs, max_new_tokens=150,
                temperature=0.1, do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
        fn_name, _ = parse_tool_call(response)
        no_fn = fn_name is None
        correct += int(no_fn)
        print(f"  '{tc['input'][:40]}' → fn={fn_name or 'none'} {'✅' if no_fn else '❌'}")
    return round(correct / len(cases), 4)

def evaluate_multiturn(model, tokenizer, cases):
    correct = 0
    for tc in cases:
        messages = [{"role": "system", "content": SYSTEM_PROMPT}] + tc["turns"]
        inputs = tokenizer.apply_chat_template(
            messages, tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt"
        ).to("cuda")
        with torch.no_grad():
            outputs = model.generate(
                input_ids=inputs, max_new_tokens=150,
                temperature=0.1, do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
        fn_name, _ = parse_tool_call(response)
        is_correct = fn_name == tc["expected_function"]
        correct += int(is_correct)
        print(f"  [{tc['id']}] expected={tc['expected_function']} got={fn_name or 'none'} {'✅' if is_correct else '❌'}")
    return round(correct / len(cases), 4)

print("All metric functions ready!")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Sentence transformer loaded!
All metric functions ready!


In [ ]:
from unsloth import FastLanguageModel
import time, torch

SYSTEM_PROMPT = (
    "You are an AI shopping assistant for an online clothing store. "
    "You can help customers: browse products, check availability, place orders, "
    "confirm payment, add delivery details, track orders, cancel orders, "
    "update orders, and process returns. "
    "Always call check_product_availability before creating any order. "
    "If the customer does not specify color or size, ask for them before proceeding. "
    "Keep responses short and clear."
)

BITEXT_SYSTEM = (
    "You are a helpful customer support assistant for an online clothing store. "
    "Help customers with their questions and issues politely and professionally."
)

VALID_FUNCTIONS = {
    "search_products",
    "check_product_availability",
    "create_order",
    "confirm_payment",
    "add_delivery_details",
    "confirm_order",
    "get_current_order_context",
    "track_order",
    "cancel_order",
    "update_order_item",
    "get_order_history",
    "create_return_request",
}


def get_system_for(tc):
    if tc.get("category") == "customer_support":
        return BITEXT_SYSTEM
    return SYSTEM_PROMPT


def build_messages(tc):
    sys = get_system_for(tc)
    if "turns" in tc:
        return [{"role": "system", "content": sys}] + tc["turns"]
    else:
        return [
            {"role": "system", "content": sys},
            {"role": "user",   "content": tc["input"]},
        ]


def generate_response(model, tokenizer, messages, max_new_tokens=200):
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    n_tokens = outputs.shape[1] - inputs.shape[1]
    return response, n_tokens


def evaluate_consistency(model, tokenizer, cases, n_runs=3, temperature=0.7):
    scores = []
    for tc in cases:
        fn_results = []
        for _ in range(n_runs):
            messages = build_messages(tc)
            ids = tokenizer.apply_chat_template(
                messages, tokenize=True,
                add_generation_prompt=True, return_tensors="pt"
            ).to(model.device)
            with torch.no_grad():
                out = model.generate(
                    input_ids=ids, max_new_tokens=150,
                    do_sample=True, temperature=temperature, top_p=0.9,
                    pad_token_id=tokenizer.eos_token_id,
                )
            resp = tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)
            fn_name, _ = parse_tool_call(resp)
            fn_results.append(fn_name == tc["expected_function"])
        consistency = sum(fn_results) / n_runs
        scores.append(consistency)
        print(f"  {tc['input'][:45]!r} -> {fn_results} ({consistency:.0%})")
    return round(sum(scores) / len(scores), 4)


def evaluate_out_of_scope(model, tokenizer, cases):
    correct = 0
    for tc in cases:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": tc["input"]}
        ]
        resp, _ = generate_response(model, tokenizer, messages, 150)
        fn_name, _ = parse_tool_call(resp)
        no_fn = fn_name is None
        correct += int(no_fn)
        print(f"  '{tc['input'][:45]}' → {'✅ no tool' if no_fn else f'❌ called {fn_name}'}")
    return round(correct / len(cases), 4)


def evaluate_multiturn(model, tokenizer, cases):
    correct = 0
    for tc in cases:
        messages = build_messages(tc)
        resp, _ = generate_response(model, tokenizer, messages, 150)
        fn_name, _ = parse_tool_call(resp)
        is_correct = fn_name == tc["expected_function"]
        correct += int(is_correct)
        print(f"  [{tc['id']}] expected={tc['expected_function']:<30} got={fn_name or 'none':<30} {'✅' if is_correct else '❌'}")
    return round(correct / len(cases), 4)


def evaluate_edge_cases(model, tokenizer, cases):
    results = []
    for tc in cases:
        messages = build_messages(tc)
        resp, _ = generate_response(model, tokenizer, messages, 150)
        fn_name, _ = parse_tool_call(resp)
        called_tool = fn_name is not None
        if tc["should_ask_clarification"]:
            correct = not called_tool
        else:
            correct = called_tool == tc["should_call_tool"]
        results.append(correct)
        print(f"  [{tc['id']}] '{tc['input'][:40]}' → tool={called_tool} {'✅' if correct else '❌'}")
    return round(sum(results) / len(results), 4)


def evaluate_model(model_name, test_cases, hf_token):
    print(f"\n{'='*65}")
    print(f"  EVALUATING: {model_name}")
    print(f"{'='*65}")

    torch.cuda.reset_peak_memory_stats()
    load_start = time.time()

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name,
        max_seq_length=2048,        
        load_in_4bit=True,
        token=hf_token,
    )
    FastLanguageModel.for_inference(model)
    load_time = round(time.time() - load_start, 1)
    print(f"  Load: {load_time}s | GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB\n")

    results = []
    all_hypotheses, all_references = [], []
    cat_results = {}
    hallucinated_names = []  

    for tc in test_cases:
        messages = build_messages(tc)
        gen_start = time.time()
        response, n_tokens = generate_response(model, tokenizer, messages)
        gen_time  = round(time.time() - gen_start, 3)
        tok_per_s = round(n_tokens / gen_time, 1) if gen_time > 0 else 0

        called_fn, called_params = parse_tool_call(response)
        expected_fn = tc["expected_function"]

        hallucinated_fn = int(
            called_fn is not None and called_fn not in VALID_FUNCTIONS
        )
        if hallucinated_fn:
            hallucinated_names.append((tc["id"], called_fn))

        if expected_fn is not None:
            fn_correct = int(called_fn == expected_fn)
            json_valid = int(called_fn is not None)
            param_acc  = check_params(called_params, tc["expected_params"])
            no_halluc  = 1
        else:
            fn_correct = int(called_fn is None)
            json_valid = 1
            param_acc  = 1.0
            no_halluc  = int(called_fn is None)

        ref     = tc["reference_response"]
        rouge_l = compute_rouge_l(response, ref)
        bleu4   = compute_bleu4(response, ref)
        sem_sim = compute_semantic_similarity(response, ref)

        all_hypotheses.append(response)
        all_references.append(ref)

        log_input = tc.get("input") or tc["turns"][-1].get("content", "")[:80]

        r = {
            "id": tc["id"], "category": tc["category"],
            "input": log_input, "response": response,
            "expected_fn": expected_fn, "called_fn": called_fn,
            "fn_correct": fn_correct, "json_valid": json_valid,
            "param_acc": param_acc, "no_hallucination": no_halluc,
            "hallucinated_fn": hallucinated_fn,
            "rouge_l": rouge_l, "bleu4": bleu4, "sem_sim": sem_sim,
            "tokens": n_tokens, "latency_s": gen_time, "tokens_per_sec": tok_per_s,
            "is_multiturn": "turns" in tc,
            "system_used": "BITEXT" if tc.get("category") == "customer_support" else "ECOMMERCE",
        }
        results.append(r)
        cat_results.setdefault(tc["category"], []).append(r)

        status = "✅" if fn_correct else "❌"
        halluc_marker = " 👻" if hallucinated_fn else ""
        mt_marker = "M" if "turns" in tc else "S"
        sys_marker = "B" if tc.get("category") == "customer_support" else "E"
        print(f"  [{tc['id']:>2}{mt_marker}{sys_marker}] {status}{halluc_marker} {tc['category']:<27} | "
              f"expected={str(expected_fn or 'none'):<30} got={str(called_fn or 'none'):<30} | {tok_per_s} tok/s")

        if not fn_correct:
            preview = response[:180].replace("\n", " ")
            print(f"        🔍 GENERATED: {preview!r}")

    print("\n  Computing BERTScore...")
    _, _, F1 = bert_score(all_hypotheses, all_references,
                          lang="en", verbose=False, device="cuda")
    for i, r in enumerate(results):
        r["bert_score"] = round(F1.tolist()[i], 4)

    print("\n  🔄 Consistency Test:")
    consistency  = evaluate_consistency(model, tokenizer, CONSISTENCY_CASES, n_runs=3)
    print(f"\n  🚫 Out-of-scope Detection:")
    out_of_scope = evaluate_out_of_scope(model, tokenizer, OUT_OF_SCOPE_CASES)
    print(f"\n  💬 Multi-turn Context:")
    multiturn    = evaluate_multiturn(model, tokenizer, MULTITURN_CASES)
    print(f"\n  ⚠️  Edge Cases:")
    edge_score   = evaluate_edge_cases(model, tokenizer, EDGE_CASES)

    if hallucinated_names:
        print(f"\n  👻 HALLUCINATED FUNCTION NAMES ({len(hallucinated_names)}):")
        for tid, fname in hallucinated_names:
            print(f"     test #{tid:>2}: '{fname}'")
    else:
        print(f"\n  ✅ No hallucinated function names")

    def avg(lst, key): return round(sum(r[key] for r in lst) / len(lst), 4) if lst else 0

    fn_cases = [r for r in results if r["expected_fn"] is not None]
    cs_cases = [r for r in results if r["expected_fn"] is None]

    per_category = {}
    for cat, cat_res in cat_results.items():
        per_category[cat] = round(avg(cat_res, "fn_correct"), 4)

    summary = {
        "model":               model_name.split("/")[-1],
        "val_loss":            None,
        "perplexity":          None,
        "fn_calling_acc_%":    round(avg(fn_cases, "fn_correct") * 100, 1),
        "json_validity_%":     round(avg(fn_cases, "json_valid") * 100, 1),
        "param_accuracy_%":    round(avg(fn_cases, "param_acc") * 100, 1),
        "no_hallucination_%":  round(avg(cs_cases, "no_hallucination") * 100, 1),
        "hallucinated_fn_%":   round(avg(results, "hallucinated_fn") * 100, 1),
        "consistency_%":       round(consistency * 100, 1),
        "out_of_scope_%":      round(out_of_scope * 100, 1),
        "multiturn_acc_%":     round(multiturn * 100, 1),
        "edge_case_%":         round(edge_score * 100, 1),
        "rouge_l":             round(avg(results, "rouge_l"), 4),
        "bleu4":               round(avg(results, "bleu4"), 4),
        "bert_score":          round(avg(results, "bert_score"), 4),
        "sem_similarity":      round(avg(results, "sem_sim"), 4),
        "avg_response_tokens": round(avg(results, "tokens"), 1),
        "avg_latency_s":       round(avg(results, "latency_s"), 3),
        "avg_tokens_per_sec":  round(avg(results, "tokens_per_sec"), 1),
        "peak_gpu_gb":         round(torch.cuda.max_memory_allocated() / 1e9, 2),
        "load_time_s":         load_time,
        "per_category":        per_category,
        "hallucinated_names":  hallucinated_names,
        "results":             results,
    }

    del model
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    return summary

print("evaluate_model() ready ✓")
print(f"SYSTEM_PROMPT (E)")
print(f"BITEXT_SYSTEM (B)")
print(f"VALID_FUNCTIONS — {len(VALID_FUNCTIONS)}")



In [ ]:
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")

# Val losses  
# Qwen v2:  best checkpoint step 6600, val_loss = 0.438194
# Llama v3: best checkpoint,            val_loss = 0.389963
VAL_LOSSES = {
    "qwen2.5-1.5b-ecommerce-v2":  0.438194,
    "llama-3.2-1b-ecommerce-v3":  0.389963,
}

MODELS = [
    "manoilokate/qwen2.5-1.5b-ecommerce-v2",
    "manoilokate/llama-3.2-1b-ecommerce-v3",
]

all_summaries = []
for model_name in MODELS:
    summary = evaluate_model(model_name, TEST_CASES, HF_TOKEN)
    key = summary["model"]
    vl  = VAL_LOSSES.get(key)
    summary["val_loss"]   = vl
    summary["perplexity"] = round(__import__("math").exp(vl), 4) if vl else None
    all_summaries.append(summary)
    print(f"\n✅ Done: {key}")

print("\nAll models evaluated!")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import json

print("=" * 72)
print("COMPLETE COMPARISON TABLE")
print("=" * 72)

rows = []
for s in all_summaries:
    rows.append({
        "Model":                 s["model"],
        "Val Loss ↓":            s["val_loss"],
        "Perplexity ↓":          s["perplexity"],
        "Fn Calling ↑ (%)":      s["fn_calling_acc_%"],
        "JSON Valid ↑ (%)":      s["json_validity_%"],
        "Param Acc ↑ (%)":       s["param_accuracy_%"],
        "No Halluc ↑ (%)":       s["no_hallucination_%"],
        "Halluc Fn Name ↓ (%)":  s["hallucinated_fn_%"],   
        "Consistency ↑ (%)":     s["consistency_%"],
        "Out-of-scope ↑ (%)":    s["out_of_scope_%"],
        "Multi-turn ↑ (%)":      s["multiturn_acc_%"],
        "Edge Cases ↑ (%)":      s["edge_case_%"],
        "ROUGE-L ↑":             s["rouge_l"],
        "BLEU-4 ↑":              s["bleu4"],
        "BERTScore ↑":           s["bert_score"],
        "Sem.Sim ↑":             s["sem_similarity"],
        "Resp. Tokens":          s["avg_response_tokens"],
        "Speed ↑ (tok/s)":       s["avg_tokens_per_sec"],
        "Latency ↓ (s)":         s["avg_latency_s"],
        "GPU ↓ (GB)":            s["peak_gpu_gb"],
    })

df = pd.DataFrame(rows).set_index("Model")
print(df.T.to_string())

print("\n" + "=" * 72)
print("ACCURACY PER CATEGORY")
print("=" * 72)
all_cats = sorted({cat for s in all_summaries for cat in s["per_category"]})
cat_rows = []
for s in all_summaries:
    row = {"Model": s["model"]}
    for cat in all_cats:
        row[cat] = f"{s['per_category'].get(cat, 0)*100:.0f}%"
    cat_rows.append(row)
df_cat = pd.DataFrame(cat_rows).set_index("Model")
print(df_cat.T.to_string())

print("\n" + "=" * 72)
print("HALLUCINATED FUNCTION NAMES (per model)")
print("=" * 72)
for s in all_summaries:
    names = s.get("hallucinated_names", [])
    print(f"\n  {s['model']}:")
    if names:
        from collections import Counter
        ctr = Counter(fname for _, fname in names)
        for fname, cnt in ctr.most_common():
            print(f"     '{fname}' × {cnt}")
    else:
        print(f"     (none)")

colors   = ["#4C72B0", "#DD8452"]
models   = [s["model"] for s in all_summaries]
fig      = plt.figure(figsize=(20, 22))
fig.suptitle("Model Comparison: Qwen2.5-1.5B vs Llama-3.2-1B\n(Fine-tuned on E-commerce Function Calling)",
             fontsize=15, fontweight="bold", y=0.98)

ax1 = fig.add_subplot(3, 3, 1, polar=True)
radar_metrics = ["fn_calling_acc_%", "json_validity_%", "no_hallucination_%",
                 "consistency_%", "out_of_scope_%", "multiturn_acc_%", "edge_case_%"]
radar_labels  = ["Fn Calling", "JSON Valid", "No Halluc",
                 "Consistency", "Out-of-scope", "Multi-turn", "Edge Cases"]
N = len(radar_metrics)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

for i, s in enumerate(all_summaries):
    vals = [s[m] for m in radar_metrics] + [s[radar_metrics[0]]]
    ax1.plot(angles, vals, "o-", linewidth=2, color=colors[i], label=s["model"].split("-")[0].upper())
    ax1.fill(angles, vals, alpha=0.1, color=colors[i])
ax1.set_xticks(angles[:-1])
ax1.set_xticklabels(radar_labels, size=8)
ax1.set_ylim(0, 100)
ax1.set_title("Behavioural Metrics (%)", size=10, pad=15)
ax1.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1), fontsize=8)

ax2 = fig.add_subplot(3, 3, 2)
x = np.arange(len(all_cats))
w = 0.35
for i, s in enumerate(all_summaries):
    vals = [s["per_category"].get(c, 0) * 100 for c in all_cats]
    ax2.bar(x + i*w - w/2, vals, w, label=s["model"].split("-")[0].upper(), color=colors[i], alpha=0.85)
ax2.set_xticks(x)
ax2.set_xticklabels([c.replace("_", "\n") for c in all_cats], fontsize=7)
ax2.set_ylabel("Accuracy (%)")
ax2.set_title("Accuracy per Category")
ax2.set_ylim(0, 110)
ax2.legend(fontsize=8)
ax2.axhline(100, color="gray", linestyle="--", alpha=0.4)
for container in ax2.containers:
    ax2.bar_label(container, fmt="%.0f%%", fontsize=6, padding=2)

ax3 = fig.add_subplot(3, 3, 3)
nlp_metrics = ["rouge_l", "bleu4", "bert_score", "sem_similarity"]
nlp_labels  = ["ROUGE-L", "BLEU-4", "BERTScore", "Sem.Sim"]
x = np.arange(len(nlp_labels))
for i, s in enumerate(all_summaries):
    vals = [s[m] for m in nlp_metrics]
    ax3.bar(x + i*w - w/2, vals, w, label=s["model"].split("-")[0].upper(), color=colors[i], alpha=0.85)
ax3.set_xticks(x)
ax3.set_xticklabels(nlp_labels)
ax3.set_title("NLP Quality Metrics")
ax3.set_ylim(0, 1.1)
ax3.legend(fontsize=8)
for container in ax3.containers:
    ax3.bar_label(container, fmt="%.3f", fontsize=7, padding=2)

ax4 = fig.add_subplot(3, 3, 4)
x = np.arange(2)
speed_vals = [[s["avg_tokens_per_sec"] for s in all_summaries],
              [s["avg_latency_s"] for s in all_summaries]]
ax4b = ax4.twinx()
b1 = ax4.bar(np.arange(len(models)) - 0.2, [s["avg_tokens_per_sec"] for s in all_summaries],
             0.35, color=colors, alpha=0.85, label="tok/s")
b2 = ax4b.bar(np.arange(len(models)) + 0.2, [s["avg_latency_s"] for s in all_summaries],
              0.35, color=colors, alpha=0.4, hatch="//", label="latency (s)")
ax4.set_ylabel("Tokens / sec ↑", color="black")
ax4b.set_ylabel("Latency (s) ↓", color="gray")
ax4.set_xticks(np.arange(len(models)))
ax4.set_xticklabels([s["model"].split("-")[0].upper() for s in all_summaries])
ax4.set_title("Speed & Latency")
ax4.bar_label(b1, fmt="%.1f", fontsize=8, padding=2)
ax4b.bar_label(b2, fmt="%.3f", fontsize=8, padding=2)

ax5 = fig.add_subplot(3, 3, 5)
bars = ax5.bar([s["model"].split("-")[0].upper() for s in all_summaries],
               [s["peak_gpu_gb"] for s in all_summaries],
               color=colors, alpha=0.85, width=0.4)
ax5.set_ylabel("Peak GPU (GB)")
ax5.set_title("GPU Memory Usage ↓")
ax5.set_ylim(0, max(s["peak_gpu_gb"] for s in all_summaries) * 1.3)
ax5.bar_label(bars, fmt="%.2f GB", fontsize=9, padding=3)

ax6 = fig.add_subplot(3, 3, 6)
beh_metrics = ["consistency_%", "out_of_scope_%", "multiturn_acc_%",
               "edge_case_%", "hallucinated_fn_%"]
beh_labels  = ["Consistency", "Out-of-scope", "Multi-turn",
               "Edge Cases", "Halluc Fn ↓"]
x = np.arange(len(beh_labels))
for i, s in enumerate(all_summaries):
    vals = [s[m] for m in beh_metrics]
    ax6.bar(x + i*w - w/2, vals, w, label=s["model"].split("-")[0].upper(), color=colors[i], alpha=0.85)
ax6.set_xticks(x)
ax6.set_xticklabels(beh_labels, fontsize=8)
ax6.set_ylabel("Score (%)")
ax6.set_title("Behavioural Tests")
ax6.set_ylim(0, 115)
ax6.legend(fontsize=8)
ax6.axhline(100, color="gray", linestyle="--", alpha=0.4)
for container in ax6.containers:
    ax6.bar_label(container, fmt="%.0f%%", fontsize=7, padding=2)

ax7 = fig.add_subplot(3, 1, 3)

W = {
    "fn_calling_acc_%":   0.25,
    "param_accuracy_%":   0.10,
    "no_hallucination_%": 0.10,
    "consistency_%":      0.15,
    "out_of_scope_%":     0.10,
    "multiturn_acc_%":    0.10,
    "edge_case_%":        0.05,
    "bert_score":         0.10, 
    "avg_tokens_per_sec": 0.05,
}

for s in all_summaries:
    s["overall_score"] = round(
        s["fn_calling_acc_%"]   * W["fn_calling_acc_%"]   +
        s["param_accuracy_%"]   * W["param_accuracy_%"]   +
        s["no_hallucination_%"] * W["no_hallucination_%"] +
        s["consistency_%"]      * W["consistency_%"]      +
        s["out_of_scope_%"]     * W["out_of_scope_%"]     +
        s["multiturn_acc_%"]    * W["multiturn_acc_%"]    +
        s["edge_case_%"]        * W["edge_case_%"]        +
        s["bert_score"] * 100   * W["bert_score"]         +
        min(s["avg_tokens_per_sec"], 100) * W["avg_tokens_per_sec"],
        2
    )

bars = ax7.barh([s["model"] for s in all_summaries],
                [s["overall_score"] for s in all_summaries],
                color=colors, alpha=0.85, height=0.4)
ax7.set_xlabel("Overall Score (weighted)")
ax7.set_title("Overall Score", fontsize=12, fontweight="bold")
ax7.set_xlim(0, 105)
for bar, s in zip(bars, all_summaries):
    ax7.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
             f"{s['overall_score']:.1f}", va="center", fontsize=11, fontweight="bold")

best = max(all_summaries, key=lambda x: x["overall_score"])
ax7.set_title(f"Overall Score  |  🏆 Best: {best['model']}", fontsize=12, fontweight="bold")

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig("/kaggle/working/model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved: /kaggle/working/model_comparison.png")

print("\n" + "=" * 72)
print("CONCLUSIONS")
print("=" * 72)

higher = ["fn_calling_acc_%", "json_validity_%", "param_accuracy_%",
          "no_hallucination_%", "consistency_%", "out_of_scope_%",
          "multiturn_acc_%", "edge_case_%", "rouge_l", "bleu4", "bert_score", "sem_similarity"]
lower  = ["val_loss", "perplexity", "avg_latency_s", "peak_gpu_gb", "hallucinated_fn_%"]

for m in higher:
    best = max(all_summaries, key=lambda x: x.get(m) or 0)
    vals = " | ".join(f"{s['model'].split('-')[0]}={s.get(m)}" for s in all_summaries)
    print(f"  ↑ {m:<28}: 🏆 {best['model'].split('-')[0]:<8}  [{vals}]")

for m in lower:
    valid = [s for s in all_summaries if s.get(m) is not None]
    if valid:
        best = min(valid, key=lambda x: x[m])
        vals = " | ".join(f"{s['model'].split('-')[0]}={s.get(m)}" for s in all_summaries)
        print(f"  ↓ {m:<28}: 🏆 {best['model'].split('-')[0]:<8}  [{vals}]")

print(f"\n{'─'*72}")
print(f"  OVERALL SCORES:")
for s in sorted(all_summaries, key=lambda x: x["overall_score"], reverse=True):
    print(f"    {s['model']:<45}: {s['overall_score']:.1f}")
print(f"\n  🏆 BEST OVERALL: {best['model']}")

save_data = [{k: v for k, v in s.items() if k != "results"} for s in all_summaries]
with open("/kaggle/working/full_evaluation.json", "w") as f:
    json.dump(save_data, f, indent=2)
df.to_csv("/kaggle/working/comparison_table.csv")
print("\nSaved: full_evaluation.json, comparison_table.csv, model_comparison.png")
